# Andvari Invocations With Blocked or Denied Egress

Loads `exports/heimdall-analysis-20260520T203812Z/tables/andvari_invocations.csv` and displays the rows where `proxy_denied_count > 0`, `blocked_egress_line_count > 0`, or both.

In [1]:
from pathlib import Path
import csv
from html import escape

try:
    from IPython.display import HTML, display
except ModuleNotFoundError:
    HTML = None
    display = None

try:
    import pandas as pd
except ModuleNotFoundError:
    pd = None

tables_dir = Path("exports/heimdall-analysis-20260520T203812Z/tables")
andvari_path = tables_dir / "andvari_invocations.csv"
sonar_path = tables_dir / "sonar_projects.csv"

def positive_count(row, column):
    value = row.get(column) or 0
    return int(float(value))

with andvari_path.open(newline="") as handle:
    andvari_rows = list(csv.DictReader(handle))

with sonar_path.open(newline="") as handle:
    sonar_rows = list(csv.DictReader(handle))

sonar_project_keys = {
    (row["agent"], row["repo_slug"], row["variant"]): row["project_key"]
    for row in sonar_rows
}

table_columns = [
    "agent",
    "sonar_project_key",
    "proxy_denied_count",
    "blocked_egress_line_count",
]

identifier_rows = []
for row in andvari_rows:
    proxy_denied_count = positive_count(row, "proxy_denied_count")
    blocked_egress_line_count = positive_count(row, "blocked_egress_line_count")
    if proxy_denied_count <= 0 and blocked_egress_line_count <= 0:
        continue

    key = (row["agent"], row["repo_slug"], row["variant"])
    identifier_rows.append(
        {
            "agent": row["agent"],
            "sonar_project_key": sonar_project_keys.get(key, ""),
            "proxy_denied_count": proxy_denied_count,
            "blocked_egress_line_count": blocked_egress_line_count,
        }
    )

print(f"{len(identifier_rows)} rows have proxy_denied_count > 0 or blocked_egress_line_count > 0.")

if pd is not None:
    identifier_table = pd.DataFrame(identifier_rows, columns=table_columns)
    display(identifier_table)
elif display is not None:
    head = "".join(f"<th>{escape(header)}</th>" for header in table_columns)
    body = "".join(
        "<tr>" + "".join(f"<td>{escape(str(row[header]))}</td>" for header in table_columns) + "</tr>"
        for row in identifier_rows
    )
    display(HTML(f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"))
else:
    identifier_rows


13 rows have proxy_denied_count > 0 or blocked_egress_line_count > 0.


agent,sonar_project_key,proxy_denied_count,blocked_egress_line_count
codex,airbnb_native-navigation__generated_v2_codex,1,0
codex,stealthcopter_AndroidNetworkTools__generated_v2_codex,1,0
codex,stealthcopter_AndroidNetworkTools__generated_v3_codex,0,7
claude,DantSu_ESCPOS-ThermalPrinter-Android__generated_v3_claude,1,0
claude,esoxjem_MovieGuide__generated_v2_claude,1,0
claude,evant_binding-collection-adapter__generated_claude,1,0
claude,,0,4
claude,JakeWharton_RxRelay__generated_claude,4,0
claude,JakeWharton_RxRelay__generated_v3_claude,1,0
claude,macrozheng_mall-tiny__generated_claude,1,0


In [2]:
csv_output_path = Path("andvari_blocked_or_denied_sonar_projects.csv")

with csv_output_path.open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=table_columns)
    writer.writeheader()
    writer.writerows(identifier_rows)

print(f"Wrote {len(identifier_rows)} rows to {csv_output_path}")


Wrote 13 rows to andvari_blocked_or_denied_sonar_projects.csv
